# Try the personal-card-agent agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `middleware/` (customer-context seeding + tool-call logging). No `mda`
  CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.
- **Option C - test the data-visualization chart skill.** Builds the agent
  exactly as `server.py` does (same `backend`/`skills` wiring), needs a
  fuller set of env vars than A/B (see that section).

**Prerequisites**

```bash
uv sync --extra notebook
# fill AI_PLATFORM_API_KEY (and LANGSMITH_API_KEY for option B) in ../.env --
# every option below uses the same GreenNode VNG Cloud MaaS model as
# server.py, not OpenAI, so OPENAI_API_KEY is not needed here.
jupyter notebook notebooks/try_agent.ipynb
```

In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (middleware/, data/,
# instructions.md, .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("AI_PLATFORM_API_KEY"), (
    f"Set AI_PLATFORM_API_KEY in {REPO_ROOT / '.env'} first"
)

from langchain_openai import ChatOpenAI

from text_utils import extract_final_text

# Same GreenNode VNG Cloud MaaS endpoint/model server.py's _default_model()
# uses -- kept in sync manually since this notebook predates that switch and
# Option A/checklist below build their own agent without importing server.py
# (importing it would also require SERVER_API_KEY/MEMORY_ID/
# USER_PREFERENCE_STRATEGY_ID just for a quick local invoke -- see Option C).
LLM_BASE_URL = "https://maas-llm-aiplatform-hcm.api.vngcloud.vn/v1"
LLM_MODEL = os.environ.get("LLM_MODEL") or "z-ai/glm-5.2-hackathon"


def default_model() -> ChatOpenAI:
    return ChatOpenAI(base_url=LLM_BASE_URL, api_key=os.environ["AI_PLATFORM_API_KEY"], model=LLM_MODEL)


## Option A - quick local invoke (no `mda dev` needed)

In [ ]:
from deepagents import create_deep_agent

from middleware.audit import log_tool_calls
from middleware.seed_customer_context import SeedCustomerContext

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Thống kê số tiền tôi đã chi tiêu vào ăn uống"}
        ]
    }
)
print(extract_final_text(result["messages"][-1].content))

## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"personal-card-agent"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [ ]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "personal-card-agent"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(extract_final_text(chunk.data["messages"][-1]["content"]))

## Option C - test the data-visualization chart skill

Uses `server.build_agent()` directly (not a fresh `create_deep_agent(...)`
call like Option A) so the run actually exercises the same `backend=
CompositeBackend(...)`/`skills=[...]` wiring `server.py` uses in production --
a plain Option-A-style agent has no `skills=[...]`, so the skill would never
be loaded.

Importing `server` pulls in its full module-level env var checks (not just
`AI_PLATFORM_API_KEY`) -- make sure these are also set in `.env`:
`SERVER_API_KEY`, `MEMORY_ID`, `USER_PREFERENCE_STRATEGY_ID` (the last one
via `tools/memory_tools.py`, imported by `server.py`).

The question below explicitly asks for a `biểu đồ` (chart), echoing the
`data-visualization` skill's own description word-for-word -- the closest
thing to a guaranteed trigger this design allows (the model still decides,
there's no enforced schema). A missing/malformed chart-json block just
yields `chart is None`, it never breaks the call.

In [4]:
import server
from text_utils import extract_chart_block, extract_final_text

chart_agent = server.build_agent()  # same backend+skills wiring as production

result = chart_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Phân tích và vẽ biểu đồ so sánh chi tiêu các danh mục của tôi tháng này.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-chart", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, chart = extract_chart_block(answer)
print("ANSWER:\n", answer)
print("\nCHART:\n", chart)

Calling tool: read_file
Finished tool: read_file
Calling tool: read_file
Finished tool: read_file
Calling tool: read_file
Finished tool: read_file
ANSWER:
 Dữ liệu đã sẵn sàng. Tháng hiện tại là **8/2026** (tính đến 29/08, tháng chưa kết thúc). Tôi sẽ phân tích breakdown theo danh mục từ `monthly_spending_summary["2026-08"]`.

**Tổng chi tiêu tháng 8/2026: 14.184.000đ** (21 giao dịch), giảm đáng kể so với tháng 7 (20.814.000đ).

**Nhận xét chính:**
- **Mua sắm** chiếm tỷ trọng lớn nhất (5.130.000đ, ~36%), nhưng đã giảm so với tháng 7 (5.730.000đ).
- **Siêu thị** và **Ăn uống** lần lượt thứ hai và ba (2.870.000đ và 2.585.000đ).
- **Di chuyển** giảm mạnh nhất: từ 7.100.000đ (tháng 7) xuống chỉ 2.100.000đ — đây là nguyên nhân chính khiến tổng chi tiêu tháng 8 thấp hơn hẳn tháng 7 (↓ ~5 triệu).
- **Giải trí** ổn định ở 599.000đ (chủ yếu là các khoản định kỳ Spotify, Netflix, CGV).
- **Hoá đơn** ~900.000đ (điện, internet) — ổn định, chi phí cố định.

CHART:
 {'chart_type': 'pie', 'title': '

## Verification checklist (local, rows 1-9 and 11 only — row 10 needs Studio, see docs/superpowers/plans/2026-08-30-personal-card-agent-verification.md)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

verify_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
    checkpointer=MemorySaver(),
)

CHECKLIST_QUESTIONS = [
    "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?",
    "Giao dịch lớn nhất tuần vừa rồi là gì?",
    "Liệt kê các giao dịch trên 2 triệu trong tháng 7",
    'Danh mục nào đang "ngốn" tiền của tôi nhất?',
    "Tôi có đang trả lãi thẻ tín dụng không, và vì sao?",
    "Tôi nên dùng thẻ tín dụng hay ghi nợ cho khoản chi 5 triệu này?",
    "Ngày sao kê và ngày đến hạn thanh toán của thẻ tôi là khi nào?",
    "Tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Ngân sách của tôi là 15 triệu/tháng.",
    "Vậy tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Khoá thẻ của tôi giúp tôi.",
]
THREAD_CONFIG = {"configurable": {"thread_id": "notebook-verify-1"}}

for question in CHECKLIST_QUESTIONS:
    result = verify_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=THREAD_CONFIG,
    )
    print(f"Q: {question}\nA: {extract_final_text(result['messages'][-1].content)}\n{'-' * 60}")

print(
    "Row 10 (cross-thread durable budget memory) cannot be tested locally — "
    "run mda dev . and re-ask the budget question in a NEW Studio thread instead."
)
